# Multi-fondamentalistes : κ_i et V_i différents

**Objectif** : Comparer la dynamique du modèle de Chiarella quand les fondamentalistes
sont hétérogènes — chaque groupe a sa propre force de rappel κ_i et
son propre processus de valeur fondamentale V_i(t).

Équation modifiée :
$$dP_t = \left[\sum_i \kappa_i (V_i(t) - P_t)\right] dt + \beta\tanh(\gamma M_t)\,dt + \sigma_N\,dW_t^{(N)}$$
$$dV_i(t) = g_i\,dt + \sigma_{V_i}\,dW_t^{(i)}$$

Les bruits $dW_t^{(i)}$ sont indépendants entre groupes.

**Configurations testées :**
- **Baseline** : κ unique = 0.08, un seul V
- **Config A** : 2 groupes (fondamentalistes réactifs + investisseurs patients)
- **Config B** : 3 groupes (optimistes + pessimistes + neutres)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.model import ChiarellaModel, FundamentalistGroup, MultiKappaModel
from src.simulation import MONTHLY_PARAMS, run_multi_kappa
from src.analysis import compute_returns, return_statistics

sns.set_theme(style='whitegrid', palette='tab10')
SEED = 2024

## 1. Définition des configurations

In [ ]:
# --- Baseline ---
baseline_model = ChiarellaModel(params=MONTHLY_PARAMS, seed=SEED)
res_base = baseline_model.simulate(n_paths=1)

# --- Config A : 2 groupes ---
# Groupe 1 : fondamentalistes réactifs (κ élevé, horizon court)
# Groupe 2 : investisseurs patients (κ faible, horizon long, vol fondamentale plus faible)
groups_A = [
    FundamentalistGroup(kappa=0.15, sigma_V=0.02, g=0.0),   # réactifs
    FundamentalistGroup(kappa=0.02, sigma_V=0.01, g=0.0),   # patients
]
res_A, df_A = run_multi_kappa(groups_A, seed=SEED)

# --- Config B : 3 groupes ---
# Groupe 1 : optimistes (drift fondamental positif)
# Groupe 2 : pessimistes (drift fondamental négatif)
# Groupe 3 : neutres (drift nul, plus précis)
groups_B = [
    FundamentalistGroup(kappa=0.05, sigma_V=0.03, g=+0.001),  # optimistes
    FundamentalistGroup(kappa=0.05, sigma_V=0.03, g=-0.001),  # pessimistes
    FundamentalistGroup(kappa=0.10, sigma_V=0.01, g=0.0),     # neutres
]
res_B, df_B = run_multi_kappa(groups_B, seed=SEED)

print('Simulations terminées.')
print(f'  Baseline   : κ={MONTHLY_PARAMS.kappa}, σ_V={MONTHLY_PARAMS.sigma_V}')
print(f'  Config A   : {len(groups_A)} groupes, κ_eff={sum(g.kappa for g in groups_A):.3f}')
print(f'  Config B   : {len(groups_B)} groupes, κ_eff={sum(g.kappa for g in groups_B):.3f}')

## 2. Divergence des valeurs fondamentales V_i(t)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Config A
ax = axes[0]
ax.plot(res_A.t, res_A.P,     label='P (log-prix)', linewidth=0.9, color='black')
ax.plot(res_A.t, res_A.V_bar, label='V̄ (consensus)', linewidth=1.2, linestyle='--', color='purple')
ax.plot(res_A.t, res_A.V_all[0], label='V₁ (réactifs κ=0.15)', linewidth=0.7, alpha=0.7)
ax.plot(res_A.t, res_A.V_all[1], label='V₂ (patients κ=0.02)', linewidth=0.7, alpha=0.7)
ax.set_title('Config A — 2 groupes')
ax.set_xlabel('Temps (mois)')
ax.legend(fontsize=8)

# Config B
ax = axes[1]
ax.plot(res_B.t, res_B.P,     label='P (log-prix)', linewidth=0.9, color='black')
ax.plot(res_B.t, res_B.V_bar, label='V̄ (consensus)', linewidth=1.2, linestyle='--', color='purple')
ax.plot(res_B.t, res_B.V_all[0], label='V₁ (optimistes)', linewidth=0.7, alpha=0.7)
ax.plot(res_B.t, res_B.V_all[1], label='V₂ (pessimistes)', linewidth=0.7, alpha=0.7)
ax.plot(res_B.t, res_B.V_all[2], label='V₃ (neutres)', linewidth=0.7, alpha=0.7)
ax.set_title('Config B — 3 groupes')
ax.set_xlabel('Temps (mois)')
ax.legend(fontsize=8)

fig.suptitle('Trajectoires P vs V_i(t) et consensus V̄', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Mispricing P − V̄ en fonction du temps

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))

mispricing_base = res_base.P - res_base.V
mispricing_A    = res_A.P   - res_A.V_bar
mispricing_B    = res_B.P   - res_B.V_bar

ax.plot(res_base.t, mispricing_base, label='Baseline (κ=0.08)', linewidth=0.8, alpha=0.8)
ax.plot(res_A.t,    mispricing_A,    label='Config A (2 groupes)', linewidth=0.8, alpha=0.8)
ax.plot(res_B.t,    mispricing_B,    label='Config B (3 groupes)', linewidth=0.8, alpha=0.8)
ax.axhline(0, color='gray', linestyle='--', alpha=0.4)

ax.set_xlabel('Temps (mois)')
ax.set_ylabel('P − V̄')
ax.set_title('Mispricing au cours du temps — comparaison des configurations')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Tableau de métriques comparatif

In [ ]:
def metrics(name, P, V, label=''):
    misp = P - V
    returns = compute_returns(P)
    stats = return_statistics(returns, annualization_factor=np.sqrt(12))
    return {
        'Modèle':            name,
        'Std(P−V̄)':          round(float(np.std(misp)), 4),
        'Max|P−V̄|':          round(float(np.max(np.abs(misp))), 4),
        'Vol annualisée':    round(stats['vol'], 4),
        'Skewness':          round(stats['skewness'], 3),
        'Kurtosis excéd.':   round(stats['kurt_excess'], 3),
    }

rows = [
    metrics('Baseline (κ=0.08)',  res_base.P, res_base.V),
    metrics('Config A (2 groupes)', res_A.P, res_A.V_bar),
    metrics('Config B (3 groupes)', res_B.P, res_B.V_bar),
]
df_metrics = pd.DataFrame(rows).set_index('Modèle')

# Variations relatives par rapport au baseline
for col in ['Std(P−V̄)', 'Max|P−V̄|', 'Vol annualisée']:
    base_val = df_metrics.loc['Baseline (κ=0.08)', col]
    df_metrics[f'Δ {col}'] = ((df_metrics[col] - base_val) / base_val * 100).round(1).astype(str) + ' %'

df_metrics

## 5. Dispersion des fondamentaux : écart entre V_i au cours du temps

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 3.5))

# Config A : écart V₁ - V₂
ax = axes[0]
spread_A = res_A.V_all[0] - res_A.V_all[1]
ax.plot(res_A.t, spread_A, linewidth=0.8, color='steelblue')
ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('Temps (mois)')
ax.set_ylabel('V₁ − V₂')
ax.set_title('Config A : désaccord entre réactifs et patients')

# Config B : range(V_i) = max(V_i) - min(V_i)
ax = axes[1]
spread_B = res_B.V_all.max(axis=0) - res_B.V_all.min(axis=0)
ax.plot(res_B.t, spread_B, linewidth=0.8, color='tomato')
ax.axhline(0, color='gray', linestyle='--', alpha=0.4)
ax.set_xlabel('Temps (mois)')
ax.set_ylabel('max(V_i) − min(V_i)')
ax.set_title('Config B : spread entre optimistes et pessimistes')

fig.suptitle('Dispersion des valeurs fondamentales V_i(t)', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Conclusions

À compléter après exécution du notebook avec les résultats numériques.